In [ ]:
!pip install -q nevergrad
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh -O /tmp/ghsparse.sh && bash /tmp/ghsparse.sh ames

In [ ]:
import warnings

import matplotlib.pyplot as plt
import nevergrad
import numpy as np
import pandas as pd
import scipy.stats
import seaborn as sns
import sklearn.base
import sklearn.compose
import sklearn.inspection
import sklearn.linear_model
import sklearn.model_selection
import sklearn.pipeline
import sklearn.preprocessing
from IPython.display import display

# Nevergrad appelle COBYLA de SciPy avec une tolérance que SciPy corrige.
warnings.filterwarnings("ignore", message="COBYLA: Invalid RHOEND")

# Régression linéaire

## Régression linéaire univariée

### Introduction

La régression linéaire est très intéressante à coder soi-même : elle n'est pas trop complexe mais permet de toucher à beaucoup de concepts du machine learning.

Nous allons continuer notre travail sur les données de prix de maisons d'Ames (Iowa). Chargeons-les :

In [ ]:
maisons = pd.read_csv("ames/train.csv", index_col="Id")

In [ ]:
display(maisons)

### Colonnes utilisées

Nous n'utiliserons dans cette première partie qu'une caractéristique, `GrLivArea` (la surface habitable, en pieds carrés, comme toutes les surfaces du jeu d'Ames : 1 pied carré ≈ 0,093 m², quand la figure du cours est en m²), et `SalePrice` (le prix de vente, en dollars) comme cible.

- *Tracez `SalePrice` en fonction de `GrLivArea` avec `sns.jointplot`.*
- *Standardisez les deux colonnes avec un `StandardScaler` chacune, $\mathbf{x}_{norm} = \frac{\mathbf{x} - \bar{\mathbf{x}}}{\sigma_\mathbf{x}}$, en deux vecteurs NumPy à une dimension, `x` et `y`. Gardez les normaliseurs dans `normaliseur_x` et `normaliseur_y` : ils serviront à revenir aux unités d'origine.*

**Documentation utile**

- [`seaborn.jointplot`](https://seaborn.pydata.org/generated/seaborn.jointplot.html)
- [`sklearn.preprocessing.StandardScaler`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html)

In [ ]:
# Votre code ici

#### Conseils supplémentaires

- `sns.jointplot(x="GrLivArea", y="SalePrice", data=maisons)` : le nuage de points et la distribution de chaque colonne.
- Un `StandardScaler` attend un tableau à deux dimensions, une colonne par variable : `maisons[["GrLivArea"]].to_numpy()`, avec deux paires de crochets. `fit_transform` renvoie une colonne, dont `[:, 0]` fait un vecteur.
- Valeurs attendues : `x.shape == (1460,)`, `x.mean()` ≈ 0 et `x.std()` ≈ 1.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
sns.jointplot(x=..., y=..., data=maisons)
plt.show()

normaliseur_x = sklearn.preprocessing.StandardScaler()
x = normaliseur_x.fit_transform(maisons[[...]].to_numpy())[:, 0]

normaliseur_y = ...
y = ...
```

#### Solution

In [ ]:
sns.jointplot(x="GrLivArea", y="SalePrice", data=maisons)
plt.show()

# Standardisation de x
normaliseur_x = sklearn.preprocessing.StandardScaler()
x = normaliseur_x.fit_transform(maisons[["GrLivArea"]].to_numpy())[:, 0]

# Standardisation de y
normaliseur_y = sklearn.preprocessing.StandardScaler()
y = normaliseur_y.fit_transform(maisons[["SalePrice"]].to_numpy())[:, 0]

### Exemple de régression linéaire

Pour comprendre l'évolution jointe du prix de vente et de la surface d'une maison, on va se limiter à une hypothèse forte : on peut trouver une fonction affine qui modélise correctement la relation entre les entrées et les sorties. Pour rappel, une fonction affine est de la forme $f(x) = ax + b$.

Géométriquement, on pourra représenter cette fonction par une droite.

Par exemple, ici, on aimerait trouver l'hypothèse suivante :

In [ ]:
sns.regplot(x=x, y=y, scatter_kws=dict(alpha=0.10), line_kws=dict(color="red"))
plt.show()

Tout ça est prometteur, mais ici nous avons utilisé [`seaborn`](https://seaborn.pydata.org/) pour calculer les paramètres de l'hypothèse qui représente au mieux les données. Nous allons maintenant voir comment le faire nous-mêmes.

### Estimation des paramètres

Pour estimer $a$ et $b$, nous avons besoin de 2 éléments :

- une fonction de coût, qui nous dira pour des paramètres donnés $a$ et $b$ si l'on se débrouille bien ou non
- une méthode d'estimation des meilleurs paramètres étant donnée cette fonction de coût

### Prédictions

En prérequis au calcul de la fonction de coût, nous devons savoir calculer des prédictions.

Pour appliquer la transformation linéaire définie par $a$ et $b$ à $\mathbf{x}$, on utilise simplement la multiplication et l'addition de scalaires à un vecteur : $a\mathbf{x} + b$.

*Définissez la fonction `predit` pour qu'elle calcule la transformation linéaire donnée par $a$ et $b$.*

In [ ]:
def predit(x: np.ndarray, a: float, b: float) -> np.ndarray:
  # Votre code ici
  ...

#### Conseils supplémentaires

- NumPy applique `a * x + b` à chaque élément du vecteur `x` : pas de boucle.
- Exemple à vérifier : `predit(np.array([1, 2, 3]), 3, 2)` vaut `[5, 8, 11]`.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def predit(x: np.ndarray, a: float, b: float) -> np.ndarray:
  return ...
```

#### Solution

In [ ]:
def predit(x: np.ndarray, a: float, b: float) -> np.ndarray:
  return a * x + b


x_exemple = np.array([1, 2, 3])
y_exemple = np.array([4, 10, 3])
a_exemple = 3
b_exemple = 2

print(predit(x_exemple, a_exemple, b_exemple))

### Fonction de coût

La fonction de coût la plus utilisée est la moyenne des carrés des différences entre les prédictions de l'hypothèse, avec les paramètres courants, et les valeurs réelles, divisée par 2 (le $\frac{1}{2}$ simplifie la dérivée).

En notant $E$ la fonction de coût et $r$ le nombre d'exemples d'apprentissage, on peut écrire :

$$
  E(a, b) = \frac{1}{2r}\sum_{i=1}^r (a x_i + b - y_i)^2
$$

- *Définissez la fonction `residus`, qui renvoie le vecteur des écarts $f(x_i) - y_i$, en vous aidant de la fonction `predit`.*
- *Définissez la fonction `cout` en vous aidant de la fonction `residus`.*

In [ ]:
def residus(x: np.ndarray, y: np.ndarray, a: float, b: float) -> np.ndarray:
  # Votre code ici
  ...


def cout(x: np.ndarray, y: np.ndarray, a: float, b: float) -> float:
  # Votre code ici
  ...

#### Conseils supplémentaires

- `np.mean(v ** 2)` : la moyenne des carrés des éléments de `v`.
- Exemple à vérifier, avec `x_exemple`, `y_exemple`, $a = 3$ et $b = 2$ : résidus `[1, -2, 8]`, coût `11.5`.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def residus(x: np.ndarray, y: np.ndarray, a: float, b: float) -> np.ndarray:
  return predit(...) - ...


def cout(x: np.ndarray, y: np.ndarray, a: float, b: float) -> float:
  return np.mean(...) / 2
```

#### Solution

In [ ]:
def residus(x: np.ndarray, y: np.ndarray, a: float, b: float) -> np.ndarray:
  return predit(x, a, b) - y


print("Résidus :", residus(x_exemple, y_exemple, a_exemple, b_exemple))


def cout(x: np.ndarray, y: np.ndarray, a: float, b: float) -> float:
  return np.mean(residus(x, y, a, b) ** 2) / 2


print("Coût :", cout(x_exemple, y_exemple, a_exemple, b_exemple))

### Optimisation des paramètres

Optimiser les paramètres revient à minimiser la fonction de coût :

$$\min_{a, b} E(a, b)$$

Pour la régression linéaire, il existe une formule exacte, les moindres carrés (c'est ce que calcule `LinearRegression` de scikit-learn, dans la seconde partie). Nous ne l'utiliserons pas ici, pour deux raisons : elle coûte cher sur de très grands jeux de données, et la méthode que nous allons coder sert aussi à la régression logistique et à bien d'autres modèles (réseaux de neurones, gradient boosting…).

Nous allons utiliser la descente de gradient. Cette méthode est itérative et fait des pas successifs dans le sens opposé à la dérivée de la fonction de coût. Dans le cas de la régression linéaire, elle converge vers l'optimum global (la meilleure solution).

En pseudo-code, avec les dérivées partielles de la frame « Exemple de dérivation — Régression linéaire » du cours, l'algorithme est :

$$
\begin{aligned}
& \text{tant que ça n'a pas convergé :} \\
& \quad a \leftarrow a - \eta \frac{\partial E}{\partial a} = a - \eta \frac{1}{r}\sum_{i=1}^r (a x_i + b - y_i)\, x_i \\
& \quad b \leftarrow b - \eta \frac{\partial E}{\partial b} = b - \eta \frac{1}{r}\sum_{i=1}^r (a x_i + b - y_i) \\
\end{aligned}
$$

où $\eta$ est le pas d'apprentissage (*learning rate*).

*Implémentez la fonction `gradient`, qui renvoie les deux dérivées partielles $\frac{\partial E}{\partial a}$ et $\frac{\partial E}{\partial b}$, en vous aidant de la fonction `residus`.*

In [ ]:
def gradient(x: np.ndarray, y: np.ndarray, a: float, b: float) -> tuple[float, float]:
  # Votre code ici
  ...

#### Conseils supplémentaires

- Avec `res` le vecteur des résidus : $\frac{\partial E}{\partial a}$ est la moyenne des `res * x`, soit `(x @ res) / y.size` (`@` : le produit scalaire), et $\frac{\partial E}{\partial b}$ la moyenne de `res`.
- Exemple à vérifier : `gradient(x_exemple, y_exemple, 3, 2)` vaut `(7.0, 2.33…)`.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def gradient(x: np.ndarray, y: np.ndarray, a: float, b: float) -> tuple[float, float]:
  res = residus(x, y, a, b)
  g_a = ...
  g_b = ...
  return g_a, g_b
```

#### Solution

In [ ]:
def gradient(x: np.ndarray, y: np.ndarray, a: float, b: float) -> tuple[float, float]:
  res = residus(x, y, a, b)
  g_a = (x @ res) / y.size
  g_b = res.mean()
  return g_a, g_b


g_a, g_b = gradient(x_exemple, y_exemple, a_exemple, b_exemple)
print(f"g_a = {g_a:.2f}, g_b = {g_b:.2f}")

### Descente de gradient

*Implémentez la descente de gradient dans la fonction `descente_gradient` : partir de $a$ et $b$ tirés au hasard, puis, au plus `nb_iter` fois, les mettre à jour avec le pas `eta` ($\eta$) ; s'arrêter dès que le coût baisse de moins de `epsilon` en une itération. Elle renvoie $a$, $b$ et la liste des coûts successifs, coût initial compris.*

In [ ]:
def descente_gradient(
  x: np.ndarray, y: np.ndarray, eta: float, nb_iter: int, epsilon: float = 1e-6
) -> tuple[float, float, list[float]]:
  # Votre code ici
  ...

#### Conseils supplémentaires

- `rng = np.random.default_rng(0)` puis `a, b = rng.random(2)` : un tirage aléatoire, mais le même à chaque exécution.
- `break` sort de la boucle `for` dès que `couts[-2] - couts[-1] < epsilon`.
- Valeurs attendues pour `descente_gradient(x, y, 0.1, 1000)` : arrêt à l'itération 43, $a$ ≈ 0,71, $b$ ≈ 0 et un coût final ≈ 0,25.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def descente_gradient(
  x: np.ndarray, y: np.ndarray, eta: float, nb_iter: int, epsilon: float = 1e-6
) -> tuple[float, float, list[float]]:
  rng = np.random.default_rng(0)
  a, b = rng.random(2)
  couts = [cout(x, y, a, b)]
  for i in range(nb_iter):
    g_a, g_b = ...
    a -= ...
    b -= ...
    couts.append(...)
    if ...:
      break
  return a, b, couts
```

#### Solution

In [ ]:
def descente_gradient(
  x: np.ndarray, y: np.ndarray, eta: float, nb_iter: int, epsilon: float = 1e-6
) -> tuple[float, float, list[float]]:
  # Un tirage aléatoire, le même à chaque exécution
  rng = np.random.default_rng(0)
  a, b = rng.random(2)
  print(f"Paramètres initiaux (aléatoires) : a = {a:.2f}, b = {b:.2f}")
  couts = [cout(x, y, a, b)]
  print(f"Coût initial : {couts[0]:.3f}")
  for i in range(nb_iter):
    g_a, g_b = gradient(x, y, a, b)
    a -= eta * g_a
    b -= eta * g_b
    couts.append(cout(x, y, a, b))
    if couts[-2] - couts[-1] < epsilon:
      print(f"Arrêt : progrès insuffisant à l'itération {i}")
      break
  print(f"Paramètres finaux : a = {a:.3f}, b = {b:.3f}")
  print(f"Coût final : {couts[-1]:.3f}")
  return a, b, couts

### Vérification de notre hypothèse

Lançons la descente de gradient avec un pas $\eta = 0{,}1$ et au plus 1 000 itérations : les coûts successifs doivent décroître.

In [ ]:
a, b, couts = descente_gradient(x, y, 0.1, 1000)
print("Premiers coûts :", np.round(couts[:5], 3))

Vérifions maintenant le modèle appris, ramené aux unités d'origine (dollars et pieds carrés), avec quelques tracés :

- valeurs prédites contre les valeurs réelles
- résidus
- coûts d'entraînement

In [ ]:
inv_x = normaliseur_x.inverse_transform(x[:, None])[:, 0]
inv_y = normaliseur_y.inverse_transform(y[:, None])[:, 0]
inv_yh = normaliseur_y.inverse_transform(predit(x, a, b)[:, None])[:, 0]

xs = np.arange(0, 6_000, 100)[:, None]
ys = normaliseur_y.inverse_transform(predit(normaliseur_x.transform(xs), a, b))
plt.plot(inv_x, inv_y, "b.", alpha=0.10)
plt.plot(xs, ys, "r")
plt.title("Prédictions et valeurs réelles")
plt.xlabel("Surface habitable (pieds carrés)")
plt.ylabel("Prix de vente ($)")
plt.show()

plt.plot(inv_x, inv_yh - inv_y, "r.", alpha=0.10)
plt.title("Résidus")
plt.xlabel("Surface habitable (pieds carrés)")
plt.ylabel("Résidus ($)")
plt.show()

plt.plot(range(len(couts)), couts)
plt.title("Coût pendant la descente de gradient")
plt.xlabel("Itérations")
plt.ylabel("Coût E")
plt.show()

### Correction de l'asymétrie positive

Les résidus s'écartent de plus en plus quand la surface augmente, et la distribution des prix est asymétrique : une longue traîne vers les maisons chères, qui tirent la droite. On peut corriger cette asymétrie positive de la variable de sortie en prenant son logarithme, [`np.log1p`](https://numpy.org/doc/stable/reference/generated/numpy.log1p.html) ($\log(1 + y)$), qui resserre la traîne et rapproche la distribution d'une loi normale ; sa transformation inverse, [`np.expm1`](https://numpy.org/doc/stable/reference/generated/numpy.expm1.html), ramène les prédictions en dollars.

La cellule suivante compare les deux distributions (avec la loi normale la plus proche, en noir), puis définit `Y`, le log-prix standardisé.

In [ ]:
def trace_avec_loi_normale(valeurs: np.ndarray) -> None:
  valeurs = np.ravel(valeurs)
  sns.histplot(valeurs, stat="density", kde=True)
  xs = np.linspace(valeurs.min(), valeurs.max(), 200)
  plt.plot(xs, scipy.stats.norm.pdf(xs, *scipy.stats.norm.fit(valeurs)), "k")


trace_avec_loi_normale(maisons["SalePrice"].to_numpy())
plt.title("Distribution de SalePrice")
plt.show()

normaliseur_Y_log = sklearn.preprocessing.StandardScaler()
Y = normaliseur_Y_log.fit_transform(np.log1p(maisons[["SalePrice"]].to_numpy()))[:, 0]

trace_avec_loi_normale(Y)
plt.title("Distribution de log(1 + SalePrice), standardisé")
plt.show()

Reprenons la descente de gradient sur `Y`, puis les mêmes tracés, en dollars :

In [ ]:
a, b, couts = descente_gradient(x, Y, 0.01, 5000)

In [ ]:
inv_Y = np.expm1(normaliseur_Y_log.inverse_transform(Y[:, None])[:, 0])
inv_Yh = np.expm1(normaliseur_Y_log.inverse_transform(predit(x, a, b)[:, None])[:, 0])
Ys = np.expm1(
  normaliseur_Y_log.inverse_transform(predit(normaliseur_x.transform(xs), a, b))
)

plt.plot(inv_x, inv_Y, "b.", alpha=0.10)
plt.plot(xs, Ys, "r")
plt.title("Prédictions et valeurs réelles")
plt.xlabel("Surface habitable (pieds carrés)")
plt.ylabel("Prix de vente ($)")
plt.show()

plt.plot(inv_x, inv_Yh - inv_Y, "r.", alpha=0.10)
plt.title("Résidus")
plt.xlabel("Surface habitable (pieds carrés)")
plt.ylabel("Résidus ($)")
plt.show()

plt.plot(range(len(couts)), couts)
plt.title("Coût pendant la descente de gradient")
plt.xlabel("Itérations")
plt.ylabel("Coût E")
plt.show()

## Régression linéaire multivariée

Passons à toutes les caractéristiques des maisons : $f(\mathbf{x}) = w_1 x_1 + \dots + w_d x_d + b = \mathbf{w} \cdot \mathbf{x} + b$, un poids $w_j$ par variable à la place de $a$ (frame « Plusieurs variables » du cours).

La fonction `pretraite` reprend un prétraitement complet des données d'Ames (et non plus seulement de la colonne `GrLivArea`) : valeurs manquantes remplies, variables ordinales converties en entiers, catégorielles encodées en indicatrices (*one-hot*). Elle renvoie, dans cet ordre, la matrice `X_train`, les prix `Y_train` (une colonne), la matrice `X_test` et le nom des colonnes, `colonnes`.

In [ ]:
def pretraite(
  fichier_ent: str, fichier_test: str
) -> tuple[np.ndarray, np.ndarray, np.ndarray, pd.Index]:
  X_train = pd.read_csv(fichier_ent, index_col="Id")
  X_test = pd.read_csv(fichier_test, index_col="Id")

  Y_train = X_train[["SalePrice"]]
  X_train = X_train.drop(columns=["SalePrice"])

  X_complet = pd.concat([X_train, X_test])

  # Remplissage par la médiane
  colonnes_1 = ["LotFrontage"]
  X_complet[colonnes_1] = X_complet[colonnes_1].fillna(X_train[colonnes_1].median())

  # Remplissage par le mode
  colonnes_2 = [
    "MSZoning",
    "Electrical",
    "KitchenQual",
    "Exterior1st",
    "Exterior2nd",
    "SaleType",
    "Utilities",
  ]
  X_complet[colonnes_2] = X_complet[colonnes_2].fillna(
    X_train[colonnes_2].mode().iloc[0, :]
  )

  # Remplissage par 0
  colonnes_4 = [
    "GarageYrBlt",
    "GarageArea",
    "GarageCars",
    "BsmtFinSF1",
    "BsmtFinSF2",
    "BsmtFullBath",
    "BsmtHalfBath",
    "BsmtUnfSF",
    "MasVnrArea",
    "TotalBsmtSF",
  ]
  X_complet[colonnes_4] = X_complet[colonnes_4].fillna(0)

  # Autres remplissages
  colonnes_5 = ["Functional"]
  X_complet[colonnes_5] = X_complet[colonnes_5].fillna("Typ")

  # On donne à tous les autres NAs la valeur string NA, qui sera une
  # catégorie
  X_complet = X_complet.fillna("NA")

  # On transforme le codage numérique en string afin que ce soit traité
  # comme une variable catégorielle
  colonnes_codes_num = ["MSSubClass"]
  X_complet[colonnes_codes_num] = X_complet[colonnes_codes_num].astype(str)

  codes_qualite = dict(NA=0, Po=1, Fa=2, TA=3, Gd=4, Ex=5)
  colonnes_qualite = [
    "BsmtCond",
    "BsmtQual",
    "ExterCond",
    "ExterQual",
    "FireplaceQu",
    "GarageCond",
    "GarageQual",
    "HeatingQC",
    "KitchenQual",
    "PoolQC",
  ]
  codes_revetement = dict(NA=0, Grvl=1, Pave=2)
  codes_finition = dict(NA=0, Unf=1, LwQ=2, Rec=3, BLQ=4, ALQ=5, GLQ=6)

  codes_ordinaux = dict(
    Alley=codes_revetement,
    BsmtExposure=dict(NA=0, No=1, Mn=2, Av=3, Gd=4),
    BsmtFinType1=codes_finition,
    BsmtFinType2=codes_finition,
    Functional=dict(Sal=1, Sev=2, Maj2=3, Maj1=4, Mod=5, Min2=6, Min1=7, Typ=8),
    LandSlope=dict(Sev=1, Mod=2, Gtl=3),
    LotShape=dict(IR3=1, IR2=2, IR1=3, Reg=4),
    PavedDrive=dict(NA=0, N=1, P=2, Y=3),
    Street=dict(Grvl=1, Pave=2),
    Utilities=dict(ELO=1, NoSeWa=2, NoSewr=3, AllPub=4),
  )

  for colonne_qualite in colonnes_qualite:
    codes_ordinaux[colonne_qualite] = codes_qualite

  # Les variables ordinales deviennent des entiers (une valeur absente du
  # dictionnaire donnerait un NaN, et astype(int) lèverait une erreur)
  for colonne, codes in codes_ordinaux.items():
    X_complet[colonne] = X_complet[colonne].map(codes).astype(int)

  print(f"Nombre de NAs : {X_complet.isnull().sum().sum()}")

  indicatrices = pd.get_dummies(X_complet)
  return (
    indicatrices.iloc[: X_train.shape[0], :].to_numpy(dtype=float),
    Y_train.to_numpy(dtype=float),
    indicatrices.iloc[X_train.shape[0] :, :].to_numpy(dtype=float),
    indicatrices.columns,
  )

In [ ]:
X_train, Y_train, X_test, colonnes = pretraite("ames/train.csv", "ames/test.csv")
X_train.shape

### Normalisation de la variable de sortie

*Standardisez les prix `Y_train` avec un `StandardScaler`, gardé dans `normaliseur_Y`, en une nouvelle variable `Y_train_std`.*

**Documentation utile**

- [`sklearn.preprocessing.StandardScaler`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html)

In [ ]:
# Votre code ici

#### Conseils supplémentaires

- Comme pour `y`, mais `Y_train` a déjà deux dimensions (une colonne) : `fit_transform(Y_train)` directement.
- Valeurs attendues : `Y_train_std.shape == (1460, 1)`, de moyenne ≈ 0.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
normaliseur_Y = ...
Y_train_std = ...
```

#### Solution

In [ ]:
normaliseur_Y = sklearn.preprocessing.StandardScaler()
Y_train_std = normaliseur_Y.fit_transform(Y_train)

### Entraînement d'une régression linéaire

- *Entraînez une `LinearRegression` (les moindres carrés) sur `X_train` et `Y_train_std`.*
- *Affichez les 10 coefficients les plus grands en valeur absolue avec le nom de leur colonne. Pourquoi ne sont-ils pas comparables entre eux tant que les caractéristiques ne sont pas standardisées ? (frame « Plusieurs variables » du cours)*
- *Prédisez le prix des deux premières maisons, en dollars, et comparez-le à leur prix réel.*

**Documentation utile**

- [`sklearn.linear_model.LinearRegression`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html)

In [ ]:
# Votre code ici

#### Conseils supplémentaires

- `regression_lineaire.coef_` a une ligne par cible : `coef_[0]` pour la seule ici.
- `pd.Series(valeurs, index=colonnes)` associe chaque poids à sa colonne ; `.abs().sort_values(ascending=False).index[:10]` donne les 10 colonnes de plus grand poids en valeur absolue.
- `normaliseur_Y.inverse_transform(...)` ramène les prédictions en dollars.
- Valeurs attendues : les plus gros poids vont à des indicatrices rares, comme `RoofMatl_ClyTile` (≈ −7,6) ; les deux premières maisons, à 208 500 et 181 500 $, sont prédites à ≈ 210 700 et 201 700 $.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
regression_lineaire = sklearn.linear_model.LinearRegression()
regression_lineaire.fit(..., ...)

coefficients = pd.Series(regression_lineaire.coef_[0], index=colonnes)
plus_grands = ...
print(coefficients[plus_grands])

predictions = normaliseur_Y.inverse_transform(regression_lineaire.predict(X_train[:2]))
print(predictions[:, 0], Y_train[:2, 0])
```

#### Solution

In [ ]:
regression_lineaire = sklearn.linear_model.LinearRegression()
regression_lineaire.fit(X_train, Y_train_std)

coefficients = pd.Series(regression_lineaire.coef_[0], index=colonnes)
plus_grands = coefficients.abs().sort_values(ascending=False).index[:10]
print(coefficients[plus_grands])

# Un poids se lit pour une hausse de 1 de sa colonne : 1 pied carré, 1 an,
# ou une indicatrice qui passe de 0 à 1. Ces unités n'ont rien de commun, et
# une indicatrice rare (quelques maisons) peut recevoir un poids énorme : seuls
# les poids de colonnes standardisées se comparent.

predictions = normaliseur_Y.inverse_transform(regression_lineaire.predict(X_train[:2]))
print("Prix prédits :", predictions[:, 0].round())
print("Prix réels :", Y_train[:2, 0])

### Validation croisée

*Écrivez une fonction `score(modele, X, Y)` qui renvoie le R² moyen d'une validation croisée à 5 plis du modèle `modele` (`cross_val_score`, `scoring="r2"`), et évaluez-la sur une `LinearRegression`.*

Le modèle est enveloppé dans un `TransformedTargetRegressor` : il standardise la cible dans chaque pli, sur les seules lignes d'entraînement de ce pli, puis ramène les prédictions en dollars. `score` prend donc les prix bruts, `Y_train`, et aucune information des lignes de validation n'entre dans l'entraînement. Le R² d'un modèle linéaire n'en change pas, mais la pénalité `alpha` des modèles régularisés, plus loin, se lit alors à l'échelle d'une cible standardisée, comme dans `Y_train_std`.

**Documentation utile**

- [`sklearn.model_selection.cross_val_score`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_val_score.html)
- [`sklearn.compose.TransformedTargetRegressor`](https://scikit-learn.org/stable/modules/generated/sklearn.compose.TransformedTargetRegressor.html)

In [ ]:
def score(
  modele: sklearn.base.BaseEstimator,
  X: np.ndarray = X_train,
  Y: np.ndarray = Y_train,
) -> float:
  # Votre code ici
  ...

#### Conseils supplémentaires

- `sklearn.compose.TransformedTargetRegressor(regressor=modele, transformer=sklearn.preprocessing.StandardScaler())`.
- `cross_val_score(estimateur, X, Y, cv=5, scoring="r2")` renvoie les 5 scores : leur moyenne avec `.mean()`.
- Valeur attendue : un R² ≈ 0,80 pour `LinearRegression`.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def score(
  modele: sklearn.base.BaseEstimator,
  X: np.ndarray = X_train,
  Y: np.ndarray = Y_train,
) -> float:
  cible_standardisee = sklearn.compose.TransformedTargetRegressor(
    regressor=..., transformer=...
  )
  scores = sklearn.model_selection.cross_val_score(...)
  return scores.mean()


print(score(sklearn.linear_model.LinearRegression()))
```

#### Solution

In [ ]:
def score(
  modele: sklearn.base.BaseEstimator,
  X: np.ndarray = X_train,
  Y: np.ndarray = Y_train,
) -> float:
  # La cible est standardisée dans chaque pli, sur les seules lignes
  # d'entraînement de ce pli.
  cible_standardisee = sklearn.compose.TransformedTargetRegressor(
    regressor=modele, transformer=sklearn.preprocessing.StandardScaler()
  )
  scores = sklearn.model_selection.cross_val_score(
    cible_standardisee, X, Y, cv=5, scoring="r2"
  )
  return scores.mean()


print(f"R² de LinearRegression : {score(sklearn.linear_model.LinearRegression()):.3f}")

### Régularisation L1 et L2

Les modèles régularisés ajoutent à l'erreur une pénalité des poids (frame « Régularisation » du cours) : `Ridge` (L2, $E + \lambda \sum_j w_j^2$) réduit les poids, `Lasso` (L1, $E + \lambda \sum_j \lvert w_j \rvert$) en annule une partie. Le $\lambda$ du cours est le paramètre `alpha` de `Lasso` ; celui de `Ridge` vaut $2r\lambda$ ($r$ lignes d'entraînement), car `Ridge` pénalise la somme des carrés des erreurs, et non leur moyenne : les `alpha` des deux modèles ne se comparent donc pas. La pénalité porte sur les poids, qui dépendent de l'unité de chaque colonne : on standardise donc les caractéristiques d'abord, dans une pipeline (`make_pipeline(StandardScaler(), Lasso(alpha=...))`), qui ajuste le `StandardScaler` sur les seules lignes d'entraînement de chaque pli de la validation croisée.

- *Pour `alpha` dans `np.logspace(-3, 1, 5)`, calculez le score d'un Lasso standardisé, `make_pipeline(StandardScaler(), Lasso(alpha=alpha))`, et du Ridge standardisé de même `alpha`.*
- *Entraînez chaque pipeline sur `X_train` et `Y_train_std`, et comptez les poids nuls de son modèle. Que fait `alpha` au Lasso ? Au Ridge ?*

**Documentation utile**

- [`sklearn.pipeline.make_pipeline`](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.make_pipeline.html)
- [`sklearn.linear_model.Lasso`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Lasso.html)
- [`sklearn.linear_model.Ridge`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Ridge.html)

In [ ]:
# Votre code ici

#### Conseils supplémentaires

- `pipeline[-1]` : le dernier pas de la pipeline, le modèle ; `pipeline[-1].coef_` : ses poids.
- `np.sum(poids == 0)` : le nombre de poids nuls.
- Valeurs attendues : le Lasso passe de 56 poids nuls (sur 244) à `alpha = 0.001` à 152 à `0.01`, où son R² est le meilleur (≈ 0,83), puis à tous à partir de `1`, où son R² tombe à 0. Le Ridge garde un R² ≈ 0,81 et un seul poids nul, celui d'une colonne constante sur l'entraînement.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
for alpha in np.logspace(-3, 1, 5):
  lasso = sklearn.pipeline.make_pipeline(
    sklearn.preprocessing.StandardScaler(), sklearn.linear_model.Lasso(alpha=alpha)
  )
  ridge = ...
  lasso.fit(X_train, Y_train_std)
  ridge.fit(...)
  print(alpha, score(lasso), np.sum(lasso[-1].coef_ == 0), ...)
```

#### Solution

In [ ]:
for alpha in np.logspace(-3, 1, 5):
  lasso = sklearn.pipeline.make_pipeline(
    sklearn.preprocessing.StandardScaler(), sklearn.linear_model.Lasso(alpha=alpha)
  )
  ridge = sklearn.pipeline.make_pipeline(
    sklearn.preprocessing.StandardScaler(), sklearn.linear_model.Ridge(alpha=alpha)
  )
  lasso.fit(X_train, Y_train_std)
  ridge.fit(X_train, Y_train_std)
  print(
    f"alpha = {alpha:g} : "
    f"Lasso R² = {score(lasso):.3f}, poids nuls = {np.sum(lasso[-1].coef_ == 0)} ; "
    f"Ridge R² = {score(ridge):.3f}, poids nuls = {np.sum(ridge[-1].coef_ == 0)}"
  )

### Recherche d'hyperparamètres

scikit-learn cherche les hyperparamètres sur une grille (`GridSearchCV`), au hasard (`RandomizedSearchCV`) ou [par divisions successives](https://scikit-learn.org/stable/modules/grid_search.html) (`HalvingGridSearchCV`, `HalvingRandomSearchCV`, encore expérimentales), qui écartent tôt les candidats médiocres. Des bibliothèques tierces explorent l'espace plus finement, en tenant compte des essais précédents : [Optuna](https://optuna.org/), la plus répandue, ou [Nevergrad](https://facebookresearch.github.io/nevergrad/), de Meta, que nous utilisons ici. Comme `alpha` varie sur plusieurs ordres de grandeur, on le cherche sur une échelle logarithmique.

- *Avec `GridSearchCV`, cherchez le meilleur `alpha` du Ridge standardisé (la pipeline de l'exercice précédent) parmi `np.logspace(-3, 3, 13)`.*
- *Écrivez une fonction `perte(alpha, X=X_train)`, l'opposé du score de ce Ridge sur `X` : elle est minimale pour le meilleur modèle.*
- *Minimisez `perte` avec Nevergrad, `alpha` sur une échelle logarithmique entre $10^{-3}$ et $10^3$, et gardez dans `meilleur_modele` le Ridge standardisé au meilleur `alpha`, `meilleur_alpha`.*

**Documentation utile**

- [`sklearn.model_selection.GridSearchCV`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html)
- [`nevergrad.p.Log`](https://facebookresearch.github.io/nevergrad/parametrization.html)
- [`nevergrad.optimizers.NGOpt`](https://facebookresearch.github.io/nevergrad/optimization.html)

In [ ]:
# Votre code ici

#### Conseils supplémentaires

- `make_pipeline` nomme chaque pas d'après sa classe, en minuscules : le paramètre `alpha` du pas `ridge` s'appelle `ridge__alpha` dans la grille, `dict(ridge__alpha=...)`. Après `recherche.fit(X_train, Y_train_std)` : `recherche.best_params_` et `recherche.best_score_`.
- Nevergrad : `nevergrad.p.Instrumentation(alpha=nevergrad.p.Log(lower=1e-3, upper=1e3))` décrit le paramètre, `nevergrad.optimizers.NGOpt(parametrization=..., budget=50)` essaie 50 valeurs, `optimiseur.minimize(perte)` renvoie la meilleure, dans `recommandation.kwargs["alpha"]`.
- Valeurs attendues : `alpha` ≈ 316 pour `GridSearchCV` et ≈ 450 pour Nevergrad, avec un R² ≈ 0,84 (contre ≈ 0,81 à `alpha = 1`).

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
ridge = sklearn.pipeline.make_pipeline(
  sklearn.preprocessing.StandardScaler(), sklearn.linear_model.Ridge()
)
recherche = sklearn.model_selection.GridSearchCV(
  ridge, dict(ridge__alpha=...), scoring="r2", cv=5
)
recherche.fit(X_train, Y_train_std)
print(recherche.best_params_, recherche.best_score_)


def perte(alpha: float, X: np.ndarray = X_train) -> float:
  ridge = ...
  return -score(ridge, X=X)


parametrage = nevergrad.p.Instrumentation(alpha=nevergrad.p.Log(...))
optimiseur = nevergrad.optimizers.NGOpt(parametrization=parametrage, budget=50)
recommandation = optimiseur.minimize(perte)
meilleur_alpha = recommandation.kwargs["alpha"]
meilleur_modele = ...
```

#### Solution

In [ ]:
ridge = sklearn.pipeline.make_pipeline(
  sklearn.preprocessing.StandardScaler(), sklearn.linear_model.Ridge()
)
recherche = sklearn.model_selection.GridSearchCV(
  ridge, dict(ridge__alpha=np.logspace(-3, 3, 13)), scoring="r2", cv=5
)
recherche.fit(X_train, Y_train_std)
alpha_grille = recherche.best_params_["ridge__alpha"]
print(f"GridSearchCV : alpha = {alpha_grille:.1f}, R² = {recherche.best_score_:.3f}")


def perte(alpha: float, X: np.ndarray = X_train) -> float:
  ridge = sklearn.pipeline.make_pipeline(
    sklearn.preprocessing.StandardScaler(), sklearn.linear_model.Ridge(alpha=alpha)
  )
  return -score(ridge, X=X)


parametrage = nevergrad.p.Instrumentation(alpha=nevergrad.p.Log(lower=1e-3, upper=1e3))
parametrage.random_state = np.random.RandomState(0)
optimiseur = nevergrad.optimizers.NGOpt(parametrization=parametrage, budget=50)
recommandation = optimiseur.minimize(perte, verbosity=0)
meilleur_alpha = recommandation.kwargs["alpha"]
print(f"Nevergrad : alpha = {meilleur_alpha:.1f}, R² = {-perte(meilleur_alpha):.3f}")

meilleur_modele = sklearn.pipeline.make_pipeline(
  sklearn.preprocessing.StandardScaler(),
  sklearn.linear_model.Ridge(alpha=meilleur_alpha),
)

### Importance des caractéristiques

L'importance par permutation d'une colonne est la baisse du score quand on mélange ses valeurs, ce qui casse son lien avec la cible. Calculée sur les lignes d'entraînement, elle mesure aussi ce que le modèle a mémorisé : on la calcule sur une partie de validation, mise de côté pendant l'entraînement.

- *Séparez `X_train` et `Y_train_std` en une partie d'apprentissage (75 %) et une de validation (25 %), et entraînez `meilleur_modele` sur la première.*
- *Calculez l'importance par permutation de chaque colonne sur la partie de validation, avec 5 répétitions, puis tracez les 10 plus grandes.*

**Documentation utile**

- [`sklearn.model_selection.train_test_split`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html)
- [`sklearn.inspection.permutation_importance`](https://scikit-learn.org/stable/modules/generated/sklearn.inspection.permutation_importance.html)

In [ ]:
# Votre code ici

#### Conseils supplémentaires

- `train_test_split(X_train, Y_train_std, test_size=0.25, random_state=0)` renvoie `X_app, X_val, Y_app, Y_val`.
- `permutation_importance(meilleur_modele, X_val, Y_val, n_repeats=5, n_jobs=-1, random_state=0)` : `n_jobs=-1` répartit le calcul sur tous les cœurs ; le résultat a un attribut `importances_mean`.
- `pd.Series(..., index=colonnes).nlargest(10).plot.bar()`.
- Valeurs attendues : `OverallQual` en tête, puis des quartiers (`Neighborhood_NridgHt`, `Neighborhood_NoRidge`), `KitchenQual` et `GrLivArea`.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
X_app, X_val, Y_app, Y_val = sklearn.model_selection.train_test_split(...)
meilleur_modele.fit(X_app, Y_app)
resultat_perm = sklearn.inspection.permutation_importance(
  meilleur_modele, ..., ..., n_repeats=5, n_jobs=-1, random_state=0
)
importances = pd.Series(resultat_perm.importances_mean, index=colonnes)
importances.nlargest(10).plot.bar()
plt.show()
```

#### Solution

In [ ]:
X_app, X_val, Y_app, Y_val = sklearn.model_selection.train_test_split(
  X_train, Y_train_std, test_size=0.25, random_state=0
)
meilleur_modele.fit(X_app, Y_app)
resultat_perm = sklearn.inspection.permutation_importance(
  meilleur_modele, X_val, Y_val, n_repeats=5, n_jobs=-1, random_state=0
)
importances = pd.Series(resultat_perm.importances_mean, index=colonnes)
importances.nlargest(10).plot.bar()
plt.title("Importance par permutation (partie de validation)")
plt.ylabel("Baisse moyenne du R²")
plt.show()

### Caractéristiques polynomiales

`PolynomialFeatures` de degré 2 ajoute à des colonnes leurs carrés et leurs produits deux à deux : le modèle reste linéaire en ses poids (frame « Une régression linéaire déguisée » du cours), mais peut suivre une courbure ou une interaction entre deux caractéristiques (la qualité et la surface, par exemple). Avec 244 colonnes, cela en ferait près de 30 000 : on se limite aux 10 plus importantes.

- *Restreignez `X_train` et `X_test` aux 10 colonnes les plus importantes, puis étendez-les par `PolynomialFeatures` de degré 2 en `X_train_poly` et `X_test_poly`.*
- *Sur `X_train_poly`, cherchez avec `GridSearchCV` le meilleur `alpha` d'un modèle L1 (`Lasso`), L2 (`Ridge`) et des deux (`ElasticNet`), standardisés, parmi `np.logspace(-3, 2, 11)`.*
- *Réglez aussi l'`alpha` du Ridge avec Nevergrad, par `perte` sur `X_train_poly`. Comparez au Ridge sur les 10 colonnes seules et sur toutes les colonnes : qu'apportent les carrés et les produits ?*

**Documentation utile**

- [`sklearn.preprocessing.PolynomialFeatures`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.PolynomialFeatures.html)
- [`sklearn.linear_model.ElasticNet`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.ElasticNet.html)

In [ ]:
# Votre code ici

#### Conseils supplémentaires

- `np.argsort(-resultat_perm.importances_mean)[:10]` : les indices des 10 colonnes les plus importantes ; `X_train[:, dix_premieres]` les sélectionne.
- `PolynomialFeatures(2, include_bias=False)` : sans la colonne constante, que l'ordonnée à l'origine du modèle remplace. `fit_transform` sur l'entraînement, `transform` sur le test.
- Un `Pipeline` aux pas nommés, `[("normalisation", StandardScaler()), ("regresseur", regresseur)]`, donne le même nom de paramètre, `regresseur__alpha`, aux trois modèles. `Lasso(max_iter=10_000)` et `ElasticNet(max_iter=10_000)` : sur ces colonnes très corrélées, les 1 000 itérations par défaut ne suffisent pas à converger.
- Valeurs attendues : 65 colonnes ; un R² ≈ 0,85 pour les trois modèles, contre ≈ 0,80 sur les 10 colonnes seules et ≈ 0,84 sur toutes.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
dix_premieres = np.argsort(-resultat_perm.importances_mean)[:10]
extension_poly = sklearn.preprocessing.PolynomialFeatures(2, include_bias=False)
X_train_poly = extension_poly.fit_transform(...)
X_test_poly = extension_poly.transform(...)

for regresseur in (sklearn.linear_model.Lasso(max_iter=10_000), ..., ...):
  modele = sklearn.pipeline.Pipeline(
    [
      ("normalisation", sklearn.preprocessing.StandardScaler()),
      ("regresseur", regresseur),
    ]
  )
  recherche = sklearn.model_selection.GridSearchCV(
    modele, dict(regresseur__alpha=...), scoring="r2", cv=5
  )
  recherche.fit(X_train_poly, Y_train_std)
  print(type(regresseur).__name__, recherche.best_params_, recherche.best_score_)


def perte_poly(alpha: float) -> float:
  return perte(alpha, X=...)
```

#### Solution

In [ ]:
dix_premieres = np.argsort(-resultat_perm.importances_mean)[:10]
extension_poly = sklearn.preprocessing.PolynomialFeatures(2, include_bias=False)
X_train_poly = extension_poly.fit_transform(X_train[:, dix_premieres])
X_test_poly = extension_poly.transform(X_test[:, dix_premieres])
print(
  f"{X_train_poly.shape[1]} colonnes : les 10, leurs 10 carrés et leurs 45 produits"
)

In [ ]:
for regresseur in (
  sklearn.linear_model.Lasso(max_iter=10_000),
  sklearn.linear_model.Ridge(),
  sklearn.linear_model.ElasticNet(max_iter=10_000),
):
  modele = sklearn.pipeline.Pipeline(
    [
      ("normalisation", sklearn.preprocessing.StandardScaler()),
      ("regresseur", regresseur),
    ]
  )
  recherche = sklearn.model_selection.GridSearchCV(
    modele, dict(regresseur__alpha=np.logspace(-3, 2, 11)), scoring="r2", cv=5
  )
  recherche.fit(X_train_poly, Y_train_std)
  alpha = recherche.best_params_["regresseur__alpha"]
  nom = type(regresseur).__name__
  print(f"{nom} : alpha = {alpha:g}, R² = {recherche.best_score_:.3f}")

In [ ]:
def perte_poly(alpha: float) -> float:
  return perte(alpha, X=X_train_poly)


parametrage = nevergrad.p.Instrumentation(alpha=nevergrad.p.Log(lower=1e-3, upper=1e3))
parametrage.random_state = np.random.RandomState(0)
optimiseur = nevergrad.optimizers.NGOpt(parametrization=parametrage, budget=50)
recommandation = optimiseur.minimize(perte_poly, verbosity=0)
alpha_poly = recommandation.kwargs["alpha"]
print(
  f"Ridge polynomial, réglé par Nevergrad : alpha = {alpha_poly:.2f}, R² = {-perte_poly(alpha_poly):.3f}"
)
print(
  f"Ridge sur les 10 colonnes seules : R² = {-perte(alpha_poly, X=X_train[:, dix_premieres]):.3f}"
)
print(f"Ridge sur toutes les colonnes : R² = {-perte(meilleur_alpha):.3f}")